In [5]:
# ============================
# 安装所有依赖
# ============================
import subprocess, sys

packages = [
    'liger_kernel',
    'datasets',
    'peft',
    'accelerate',
    'bitsandbytes',
    'sentencepiece',
    'scikit-learn',
    'pandas==2.2.2',
    'matplotlib',
    'seaborn',
    'huggingface_hub',
    'trl'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('✅ 依赖安装完成')


✅ 依赖安装完成


In [ ]:
!pip install -q "transformers==4.44.2" "trl==0.11.0"

In [6]:
# 依赖已在上方 Cell 0 安装完成，无需重复安装


In [7]:
import os
import re
import gc
import json
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ========== Config ==========
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/mono_logic_lora"

SEED = 42
MAX_LEN = 1024

TRAIN_SUBSET = 1500
EVAL_SUBSET = 200

NUM_EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACC = 8

USE_MONO_TAGS_FOR_TRAIN = True
USE_COT_IN_TRAIN_TARGET = True

LABELS = ["PROVED", "DISPROVED", "UNKNOWN"]

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [8]:
ds = load_dataset("hitachi-nlp/FLD.v2")
print(ds)

for split_name in ds.keys():
    print(split_name, ds[split_name].column_names[:20], " ... total =", len(ds[split_name]))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-81d338afabf39c(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/validation-00000-of-00001-bcc05849f(…):   0%|          | 0.00/6.40M [00:00<?, ?B/s]

data/test-00000-of-00001-9c326759ac5a453(…):   0%|          | 0.00/6.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['version', 'hypothesis', 'hypothesis_formula', 'context', 'context_formula', 'proofs', 'proofs_formula', 'negative_hypothesis', 'negative_hypothesis_formula', 'negative_proofs', 'negative_original_tree_depth', 'original_tree_depth', 'depth', 'num_formula_distractors', 'num_translation_distractors', 'num_all_distractors', 'proof_label', 'negative_proof_label', 'world_assump_label', 'negative_world_assump_label', 'prompt_serial', 'proof_serial'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['version', 'hypothesis', 'hypothesis_formula', 'context', 'context_formula', 'proofs', 'proofs_formula', 'negative_hypothesis', 'negative_hypothesis_formula', 'negative_proofs', 'negative_original_tree_depth', 'original_tree_depth', 'depth', 'num_formula_distractors', 'num_translation_distractors', 'num_all_distractors', 'proof_label', 'negative_proof_label', 'world_assump_label', 'negative_world_assump_label', 'promp

In [9]:
if "train" in ds:
    train_raw = ds["train"]
else:
    first_split = list(ds.keys())[0]
    train_raw = ds[first_split]

if "validation" in ds:
    eval_raw = ds["validation"]
elif "test" in ds:
    eval_raw = ds["test"]
else:
    split_tmp = train_raw.train_test_split(test_size=0.1, seed=SEED)
    train_raw = split_tmp["train"]
    eval_raw = split_tmp["test"]

train_raw = train_raw.shuffle(seed=SEED)
eval_raw = eval_raw.shuffle(seed=SEED)

train_raw = train_raw.select(range(min(TRAIN_SUBSET, len(train_raw))))
eval_raw = eval_raw.select(range(min(EVAL_SUBSET, len(eval_raw))))

print("Train size:", len(train_raw))
print("Eval size :", len(eval_raw))

print(train_raw[0].keys())
print(train_raw[0])

Train size: 1500
Eval size : 200
dict_keys(['version', 'hypothesis', 'hypothesis_formula', 'context', 'context_formula', 'proofs', 'proofs_formula', 'negative_hypothesis', 'negative_hypothesis_formula', 'negative_proofs', 'negative_original_tree_depth', 'original_tree_depth', 'depth', 'num_formula_distractors', 'num_translation_distractors', 'num_all_distractors', 'proof_label', 'negative_proof_label', 'world_assump_label', 'negative_world_assump_label', 'prompt_serial', 'proof_serial'])
{'version': 'DeductionInstance', 'hypothesis': 'the porterhouse does argue acrophony.', 'hypothesis_formula': '{E}{c}', 'context': 'sent1: if the fact that the birdhouse is not a LEM or is wieldy or both is false the chamomile is a Terry. sent2: if that the malt does not argue acrophony and/or it is a LEM is not right the birdhouse is low-level. sent3: if something is low-level and it is wieldy it does not argue acrophony. sent4: if that the birdhouse is not a LEM and/or is not wieldy is not true then 

In [10]:
UPWARD_CUES = [
    "all", "every", "each", "some", "a ", "an ", "many", "most", "several", "exists", "there is"
]
DOWNWARD_CUES = [
    "no", "none", "nobody", "nothing", "few", "without", "never", "neither", "nor", "at most"
]
NEGATION_CUES = [
    "not", "no", "never", "none", "nobody", "nothing", "neither", "nor", "does not", "is not", "are not"
]
CONDITIONAL_CUES = [
    "if", "unless", "only if", "implies", "then"
]

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def heuristic_mono_tags(sentence: str):
    s = " " + normalize_space(sentence.lower()) + " "
    tags = []

    found_up = [w.strip() for w in UPWARD_CUES if f" {w.strip()} " in s]
    found_down = [w.strip() for w in DOWNWARD_CUES if f" {w.strip()} " in s]
    found_neg = [w.strip() for w in NEGATION_CUES if f" {w.strip()} " in s]
    found_cond = [w.strip() for w in CONDITIONAL_CUES if f" {w.strip()} " in s]

    if found_up:
        tags.append("UPWARD=" + ",".join(sorted(set(found_up))))
    if found_down:
        tags.append("DOWNWARD=" + ",".join(sorted(set(found_down))))
    if found_neg:
        tags.append("NEGATION=" + ",".join(sorted(set(found_neg))))
    if found_cond:
        tags.append("CONDITIONAL=" + ",".join(sorted(set(found_cond))))

    if not tags:
        tags = ["NEUTRAL"]

    return "[" + " | ".join(tags) + "]"

def annotate_sentence(sentence: str):
    sentence = normalize_space(sentence)
    return f"{heuristic_mono_tags(sentence)} {sentence}"

def annotate_context(context: str):
    lines = [x for x in str(context).split("\n") if x.strip()]
    out = []
    for line in lines:
        m = re.match(r"^(sent\d+:\s*)(.*)$", line.strip(), re.I)
        if m:
            prefix, sent = m.groups()
            out.append(prefix + annotate_sentence(sent))
        else:
            out.append(annotate_sentence(line.strip()))
    return "\n".join(out)


In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

print("Model loaded.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [12]:
SYSTEM_PROMPT = """You are a careful formal reasoning assistant.
Your job is to determine whether the hypothesis is logically supported by the context.
Use only the given context.
Return one of these labels: PROVED, DISPROVED, UNKNOWN.
If reasoning is requested, end with a final line exactly in the format:
Final label: <PROVED|DISPROVED|UNKNOWN>
"""

def render_chat(messages, add_generation_prompt=False):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt
        )
    text = ""
    for m in messages:
        text += f"{m['role'].upper()}:\n{m['content']}\n\n"
    if add_generation_prompt:
        text += "ASSISTANT:\n"
    return text

def get_label(example):
    if "label" in example:
        label = str(example["label"]).upper().strip()
    elif "answer" in example:
        label = str(example["answer"]).upper().strip()
    elif "proof_label" in example:
        label = str(example["proof_label"]).upper().strip()
    else:
        label = "UNKNOWN"

    if label not in LABELS:
        if label in ["ENTAILMENT", "ENTAILED", "PROOF", "PROVED"]:
            label = "PROVED"
        elif label in ["CONTRADICTION", "DISPROVED", "REFUTED"]:
            label = "DISPROVED"
        else:
            label = "UNKNOWN"
    return label

def build_user_prompt(example, use_mono=False, ask_reasoning=True):
    context = str(example["context"])
    hypothesis = str(example["hypothesis"])

    if use_mono:
        context = annotate_context(context)
        hypothesis = annotate_sentence(hypothesis)

    prompt = f"""Task:
Determine whether the hypothesis follows from the context.

Allowed labels:
- PROVED
- DISPROVED
- UNKNOWN

Rules:
1. Use only the provided context.
2. Prefer exact logical deduction over world knowledge.
3. Do not invent facts not in the context.
"""

    if ask_reasoning:
        prompt += '4. Briefly reason, then end with: Final label: <LABEL>\n'
    else:
        prompt += '4. Output only one label.\n'

    prompt += f"""
Context:
{context}

Hypothesis:
{hypothesis}
"""
    return prompt.strip()

def build_train_text(example, use_mono=False):
    label = get_label(example)
    user_prompt = build_user_prompt(example, use_mono=use_mono, ask_reasoning=True)

    proof = ""
    if "proof_serial" in example and example["proof_serial"] is not None:
        proof = str(example["proof_serial"]).strip()
    elif "proofs" in example and example["proofs"] is not None:
        proof = str(example["proofs"]).strip()

    INVALID_PROOFS = {"__UNKNOWN__", "[]", "", "none", "null"}
    clean_proof = proof if proof.lower() not in INVALID_PROOFS else ""

    if USE_COT_IN_TRAIN_TARGET and clean_proof:
        assistant_text = f"Reasoning:\n{clean_proof}\nFinal label: {label}"
    else:
        assistant_text = f"Final label: {label}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_text},
    ]
    return {"text": render_chat(messages, add_generation_prompt=False), "label": label}

train_sft = train_raw.map(
    lambda x: build_train_text(x, use_mono=USE_MONO_TAGS_FOR_TRAIN),
    remove_columns=train_raw.column_names
)

print(train_sft[0]["text"][:2000])
print("Label:", train_sft[0]["label"])

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

<|im_start|>system
You are a careful formal reasoning assistant.
Your job is to determine whether the hypothesis is logically supported by the context.
Use only the given context.
Return one of these labels: PROVED, DISPROVED, UNKNOWN.
If reasoning is requested, end with a final line exactly in the format:
Final label: <PROVED|DISPROVED|UNKNOWN>
<|im_end|>
<|im_start|>user
Task:
Determine whether the hypothesis follows from the context.

Allowed labels:
- PROVED
- DISPROVED
- UNKNOWN

Rules:
1. Use only the provided context.
2. Prefer exact logical deduction over world knowledge.
3. Do not invent facts not in the context.
4. Briefly reason, then end with: Final label: <LABEL>

Context:
sent1: [UPWARD=a,exists,there is | NEGATION=does not,is not,not | CONDITIONAL=if,then] if the fact that the birdhouse is not a LEM or is wieldy or both is false the chamomile is a Terry. sent2: if that the malt does not argue acrophony and/or it is a LEM is not right the birdhouse is low-level. sent3: if

In [13]:
def extract_label(text: str):
    t = str(text).upper()

    m = re.search(r"FINAL LABEL\s*:\s*(PROVED|DISPROVED|UNKNOWN)", t)
    if m:
        return m.group(1)

    for lb in LABELS:
        if re.search(rf"\b{lb}\b", t):
            return lb

    return "UNKNOWN"

@torch.no_grad()
def generate_one(model, example, use_mono=False, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(example, use_mono=use_mono, ask_reasoning=True)},
    ]
    prompt = render_chat(messages, add_generation_prompt=True)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    pred = extract_label(gen_text)

    return gen_text, pred

def run_eval(model, eval_dataset, use_mono=False, tag="run"):
    rows = []
    for ex in tqdm(eval_dataset, desc=f"Evaluating {tag}"):
        gen_text, pred = generate_one(model, ex, use_mono=use_mono)
        gold = get_label(ex)

        rows.append({
            "tag": tag,
            "gold": gold,
            "pred": pred,
            "correct": int(gold == pred),
            "context": ex["context"],
            "hypothesis": ex["hypothesis"],
            "raw_output": gen_text,
        })
    return pd.DataFrame(rows)

def summarize_results(df, name):
    y_true = df["gold"].tolist()
    y_pred = df["pred"].tolist()
    return {
        "run": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "n": len(df),
    }

In [14]:
baseline_plain_df = run_eval(base_model, eval_raw, use_mono=False, tag="base_plain")
baseline_mono_df = run_eval(base_model, eval_raw, use_mono=True, tag="base_mono")

summary_rows = []
summary_rows.append(summarize_results(baseline_plain_df, "base_plain"))
summary_rows.append(summarize_results(baseline_mono_df, "base_mono"))

summary_df = pd.DataFrame(summary_rows)
summary_df

Evaluating base_plain:   0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating base_mono:   0%|          | 0/200 [00:00<?, ?it/s]

,run,accuracy,macro_f1,n
0,base_plain,0.375,0.362293,200
1,base_mono,0.410,0.391279,200


In [15]:
!cd /content/CSE_188_NLP && git pull


/bin/bash: line 1: cd: /content/CSE_188_NLP: No such file or directory


In [16]:
!pip install -q trl==0.11.0

In [18]:
from trl import SFTConfig, SFTTrainer
from trl.trainer import DataCollatorForCompletionOnlyLM

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    lr_scheduler_type="cosine",
    warmup_steps=20,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    max_grad_norm=0.3,
    # completion_only_loss=True,
    dataset_text_field="text",
    max_seq_length=MAX_LEN,
)

response_template = "<|im_start|>assistant\n"

collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    processing_class=tokenizer,
    data_collator=collator,
)

trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'processing_class'

In [ ]:
finetuned_mono_df = run_eval(model, eval_raw, use_mono=True, tag="finetuned_mono")

summary_rows.append(summarize_results(finetuned_mono_df, "finetuned_mono"))
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
all_runs = pd.concat([baseline_plain_df, baseline_mono_df, finetuned_mono_df], ignore_index=True)

summary_csv = os.path.join(OUTPUT_DIR, "summary_metrics.csv")
all_csv = os.path.join(OUTPUT_DIR, "all_predictions.csv")

summary_df.to_csv(summary_csv, index=False)
all_runs.to_csv(all_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", all_csv)

print("\n=== Classification reports ===\n")
for run_name, df_run in [
    ("base_plain", baseline_plain_df),
    ("base_mono", baseline_mono_df),
    ("finetuned_mono", finetuned_mono_df),
]:
    print(f"\n--- {run_name} ---")
    print(classification_report(df_run["gold"], df_run["pred"], labels=LABELS, digits=4))

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(8, 5))
sns.barplot(data=summary_df, x="run", y="accuracy")
plt.ylim(0, 1)
plt.title("Accuracy by Setting")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=summary_df, x="run", y="macro_f1")
plt.ylim(0, 1)
plt.title("Macro-F1 by Setting")
plt.tight_layout()
plt.show()

for run_name, df_run in [
    ("base_plain", baseline_plain_df),
    ("base_mono", baseline_mono_df),
    ("finetuned_mono", finetuned_mono_df),
]:
    cm = confusion_matrix(df_run["gold"], df_run["pred"], labels=LABELS)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.title(f"Confusion Matrix: {run_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Gold")
    plt.tight_layout()
    plt.show()

In [ ]:
from peft import PeftModel

del trainer
gc.collect()
torch.cuda.empty_cache()

base_reload = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_reload.config.use_cache = False

reload_model = PeftModel.from_pretrained(base_reload, OUTPUT_DIR)
reload_model.eval()

reload_df = run_eval(reload_model, eval_raw.select(range(min(50, len(eval_raw)))), use_mono=True, tag="reloaded_ft")
print(summarize_results(reload_df, "reloaded_ft"))